<a href="https://colab.research.google.com/github/sofianik314/MSc-Thesis-Codes-2026/blob/main/hsbs_2025_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PyMuPDF (fitz)	Text extraction

pdfplumber	Detecting tables

camelot	Extracting tables

### Method 1: Redacting the Table Area

approach # 1

Using pdfplumber - package, designed specifically for detecting tables.

In [ ]:
!pip install pymupdf pdfplumber pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 46.1 MB/s eta 0:00:00


In [ ]:
import fitz

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving 260225-annual-report-and-accounts-2025.pdf to 260225-annual-report-and-accounts-2025.pdf


In [ ]:
import fitz

doc = fitz.open("260225-annual-report-and-accounts-2025.pdf")

In [ ]:
full_text = ""

for page in doc:
    full_text += page.get_text("text")

print(full_text[:1000])

Opening up a world of opportunity 
HSBC is one of the largest banking and financial 
services organisations in the world.
Guided by our purpose of opening up a world of 
opportunity, our ambition is to become the world’s 
most trusted bank globally, putting customers at 
the heart of everything we do. 
In this year’s report
Strategic report
2  
Highlights
4  
Who we are
5  
Group Chairman’s shareholder letter 
7  
Group CEO’s shareholder letter
9  
Our strategy 
12  
Financial overview
16  Business segments
25  ESG overview
27 
Board engagement with stakeholders
  
and decision making 
30  Risk overview
32  
Long-term viability and going 
      
concern statement
Environmental, social and
governance (‘ESG’) review
34  
Environmental
52  
Social
58  
Governance
Financial review
66  
Financial summary
76  
Business segments and legal 
        entities
92  
Alternative performance measures
Risk review
98  
Our approach to risk
100  Top and emerging risks
105  Our material banking risks
Co

Removing tables & noise

In [ ]:
import re

def is_table_line(line):
    numbers = re.findall(r"\d+", line)
    return len(numbers) > 5  #Remove lines with too many numbers

cleaned_lines = []

for line in full_text.split("\n"):
    if not is_table_line(line):
        cleaned_lines.append(line)

clean_text = "\n".join(cleaned_lines)

Table-like structures were removed using a heuristic based on numeric density, where lines containing more than five numerical tokens were excluded, as they typically represent financial tables rather than narrative text

Remove repeated lines

In [ ]:
from collections import Counter

lines = full_text.split("\n")
counts = Counter(lines)

cleaned_lines = [l for l in lines if counts[l] < 5]

Use pdfplumber

In [ ]:
import pdfplumber

text_without_tables = ""

with pdfplumber.open("260225-annual-report-and-accounts-2025.pdf") as pdf:
    for page in pdf.pages:
        tables = page.extract_tables()
        table_texts = []

        for table in tables:
            for row in table:
                table_texts.append(" ".join([str(cell) for cell in row if cell]))

        page_text = page.extract_text()

        # remove table content
        for t in table_texts:
            if t in page_text:
                page_text = page_text.replace(t, "")

        text_without_tables += page_text + "\n"

In [ ]:
print(text_without_tables[:2000])


Opening up a world of opportunity
HSBC is one of the largest banking and financial
services organisations in the world.
Guided by our purpose of opening up a world of
opportunity, our ambition is to become the world’s
most trusted bank globally, putting customers at
the heart of everything we do.
In this year’s report
Strategic report Risk review This Strategic Report was approved by the
Board on 25 February 2026 and signed on its
2 Highlights 98 Our approach to risk behalf by the Group Chairman.
4 Who we are 100 Top and emerging risks
5 Group Chairman’s shareholder letter 105 Our material banking risks
7 Group CEO’s shareholder letter Brendan Nelson
9 Our strategy Corporate governance report Group Chairman
12 Financial overview
16 Business segments 187 Biographies of Directors and
25 ESG overview senior management A reminder
27 Board engagement with stakeholders 200 Board committees The currency we report in is US dollars.
and decision making 216 Directors’ remuneration report Our ap

Look at the first 15 pages after removing tables


In [ ]:
with pdfplumber.open("260225-annual-report-and-accounts-2025.pdf") as pdf:
    for i, page in enumerate(pdf.pages[:15]):  # first 15 pages only
        page_text = page.extract_text()
        print(f"\n--- PAGE {i} ---\n")
        print(page_text[:1000])


--- PAGE 0 ---



--- PAGE 1 ---

Opening up a world of opportunity
HSBC is one of the largest banking and financial
services organisations in the world.
Guided by our purpose of opening up a world of
opportunity, our ambition is to become the world’s
most trusted bank globally, putting customers at
the heart of everything we do.
In this year’s report
Strategic report Risk review This Strategic Report was approved by the
Board on 25 February 2026 and signed on its
2 Highlights 98 Our approach to risk behalf by the Group Chairman.
4 Who we are 100 Top and emerging risks
5 Group Chairman’s shareholder letter 105 Our material banking risks
7 Group CEO’s shareholder letter Brendan Nelson
9 Our strategy Corporate governance report Group Chairman
12 Financial overview
16 Business segments 187 Biographies of Directors and
25 ESG overview senior management A reminder
27 Board engagement with stakeholders 200 Board committees The currency we report in is US dollars.
and decision making 216 Dir

Why pdfplumber is not perfect:

The reason pdfplumber.find_tables() failed to remove that specific chunk is that the table in the document is likely borderless or uses a complex layout that the default settings could not detect. When a table does not have clear vertical and horizontal lines, pdfplumber does not "see" it as a table and instead treats it as regular flowing text.

Approach#2

Detecting heavy structures of text was not a good idea because table mught consist of tags and labels. Thus, we need to detect “not a real sentence”

Real sentences:

have verbs

flow naturally


Tables:

are fragments with no grammar

In [ ]:
import re

def is_table_line(line):
    numbers = re.findall(r"\d+", line)
    words = re.findall(r"[a-zA-Z]+", line)

    digit_ratio = sum(c.isdigit() for c in line) / max(len(line), 1)

    return (
        len(numbers) >= 3 or                 # many numbers
        digit_ratio > 0.3 or                 # mostly numeric
        (len(numbers) >= 2 and len(words) >= 2) or  # mixed table
        len(line.split()) > 10 or            # too many columns
        "  " in line                         # spacing pattern
    )

r: Stands for "raw string," which tells Python to treat backslashes literally (standard practice for regex).

\d: Matches any single digit (0-9).

+: This is a quantifier meaning "one or more."

In [ ]:
print(clean_text[:4000])

Opening up a world of opportunity 
HSBC is one of the largest banking and financial 
services organisations in the world.
Guided by our purpose of opening up a world of 
opportunity, our ambition is to become the world’s 
most trusted bank globally, putting customers at 
the heart of everything we do. 
In this year’s report
Strategic report
2  
Highlights
4  
Who we are
5  
Group Chairman’s shareholder letter 
7  
Group CEO’s shareholder letter
9  
Our strategy 
12  
Financial overview
16  Business segments
25  ESG overview
27 
Board engagement with stakeholders
  
and decision making 
30  Risk overview
32  
Long-term viability and going 
      
concern statement
Environmental, social and
governance (‘ESG’) review
34  
Environmental
52  
Social
58  
Governance
Financial review
66  
Financial summary
76  
Business segments and legal 
        entities
92  
Alternative performance measures
Risk review
98  
Our approach to risk
100  Top and emerging risks
105  Our material banking risks
Co

In [ ]:
with pdfplumber.open("260225-annual-report-and-accounts-2025.pdf") as pdf:
    for i, page in enumerate(pdf.pages[:15]):
        print(f"\n========== PAGE {i} ==========\n")
        print(page.extract_text()[:1000])


========== PAGE 0 ==========



========== PAGE 1 ==========

Opening up a world of opportunity
HSBC is one of the largest banking and financial
services organisations in the world.
Guided by our purpose of opening up a world of
opportunity, our ambition is to become the world’s
most trusted bank globally, putting customers at
the heart of everything we do.
In this year’s report
Strategic report Risk review This Strategic Report was approved by the
Board on 25 February 2026 and signed on its
2 Highlights 98 Our approach to risk behalf by the Group Chairman.
4 Who we are 100 Top and emerging risks
5 Group Chairman’s shareholder letter 105 Our material banking risks
7 Group CEO’s shareholder letter Brendan Nelson
9 Our strategy Corporate governance report Group Chairman
12 Financial overview
16 Business segments 187 Biographies of Directors and
25 ESG overview senior management A reminder
27 Board engagement with stakeholders 200 Board committees The currency we report in is US dollars.

Approach #2" is a heuristic-based filter, which means it uses "rules of thumb" to guess what a table looks like. The idea of detecting the actual sentences is good, however, Approach #2 uses brutal force and criteria, that seem logical to human brain while missing specific parts of the table (e.g. a table might contain a header that falls under the rules of "sentence")


Approach #4
Using ML (Computer Vision) and NLP (Grammar analysis)

1. The ML Way: Computer Vision
Instead of looking for lines, Machine Learning models look for the visual pattern of a table (rows of numbers, column alignments). The current industry standard is Table Transformer (TATR) by Microsoft or LayoutLM.

How it works:

Convert to Image: The PDF page is turned into a high-res image.

Object Detection: The model (usually a Transformer) predicts a "bounding box" (rectangle) around anything it thinks is a table.

Redaction: You take those coordinates and use your existing PyMuPDF logic to "white out" that area.

In [ ]:
!pip install transformers torch torchvision Pillow pymupdf

In [ ]:
import fitz
import torch
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForObjectDetection

def remove_tables_with_ml(input_path, output_path, max_pages=25, confidence_threshold=0.85):
    print("Loading AI Model (this takes a few seconds)...")
    # Load the image processor and the object detection model
    processor = AutoImageProcessor.from_pretrained("microsoft/table-transformer-detection")
    model = AutoModelForObjectDetection.from_pretrained("microsoft/table-transformer-detection")

    doc = fitz.open(input_path)

    num_pages = len(doc)

    for page_num in range(num_pages):
        print(f"Processing Page {page_num + 1}/{num_pages}...")
        page = doc[page_num]

        # 1. Convert PDF page to an Image
        # We use standard 72 DPI so 1 image pixel = 1 PDF point (makes math easy)
        pix = page.get_pixmap(dpi=72)
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)

        # 2. Feed the image to the AI model
        inputs = processor(images=img, return_tensors="pt")
        outputs = model(**inputs)

        # 3. Process the AI's output to get usable coordinates
        # target_sizes tells the model the dimensions of our image to scale the boxes correctly
        target_sizes = torch.tensor([img.size[::-1]])
        results = processor.post_process_object_detection(
            outputs,
            threshold=confidence_threshold,
            target_sizes=target_sizes
        )[0]

        # 4. Redact (white out) the detected tables
        for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
            # In this specific model, label '0' means 'Table'
            if label.item() == 0:
                # Convert the tensor box [x_min, y_min, x_max, y_max] to a standard Python list
                box_coords = [round(i, 2) for i in box.tolist()]

                # Create a PyMuPDF rectangle
                rect = fitz.Rect(box_coords[0], box_coords[1], box_coords[2], box_coords[3])

                # Add a white box over the table
                page.add_redact_annot(rect, fill=(1, 1, 1))
                print(f"  -> Found table on page {page_num} with {score:.2f} confidence. Redacting.")

        # Apply all the redactions we just added to this page
        page.apply_redactions()

    # Save the cleaned document
    doc.save(output_path)
    print(f"\nDone! Cleaned PDF saved to: {output_path}")

# Make sure to update the input filename to match your uploaded file!
input_file = "260225-annual-report-and-accounts-2025.pdf"
output_file = "report_cleaned_by_AI.pdf"

remove_tables_with_ml(input_file, output_file, max_pages=15)

Loading AI Model (this takes a few seconds)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/273 [00:00<?, ?B/s]

The image processor of type `DetrImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:2586: UserWarning: for conv1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  super()._load_from_state_dict(
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/batchnorm.py:133: UserWarning: for bn1.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, whi

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

TableTransformerForObjectDetection LOAD REPORT from: microsoft/table-transformer-detection
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Processing Page 1/372...
Processing Page 2/372...
Processing Page 3/372...
Processing Page 4/372...
Processing Page 5/372...
Processing Page 6/372...
Processing Page 7/372...
  -> Found table on page 6 with 0.95 confidence. Redacting.
Processing Page 8/372...
  -> Found table on page 7 with 0.88 confidence. Redacting.
  -> Found table on page 7 with 0.88 confidence. Redacting.
  -> Found table on page 7 with 0.97 confidence. Redacting.
Processing Page 9/372...
  -> Found table on page 8 with 0.87 confidence. Redacting.
Processing Page 10/372...
  -> Found table on page 9 with 0.88 confidence. Redacting.
  -> Found table on page 9 with 0.90 confidence. Redacting.
  -> Found table on page 9 with 0.95 confidence. Redacting.
  -> Found table on page 9 with 0.92 confidence. Redacting.
Processing Page 11/372...
Processing Page 12/372...
Processing Page 13/372...
Processing Page 14/372...
Processing Page 15/372...
  -> Found table on page 14 with 0.99 confidence. Redacting.
Processing Page 16

In [ ]:
from google.colab import files
files.download("report_cleaned_by_AI.pdf")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

The Limitations of Computer Vision in Document Layout Analysis

1. The Confidence Threshold Dilemma (Precision vs. Recall)
Object detection relies on a confidence score (e.g., 0.0 to 1.0) to determine if a bounded region is a table. This creates an unavoidable trade-off:

When set too strict (e.g., 0.85): The model prioritizes Precision, leading to False Negatives. It successfully extracts obvious tables but completely ignores dense, complex, or embedded tables.

When set too low (e.g., 0.60): The model prioritizes Recall, leading to severe False Positives. It becomes  over-sensitive.

2. Geometric Mimicry (The "Column" Confusion)

To a CV model, the vertical strip of whitespace (the gutter) separating these text columns looks geometrically identical to a borderless, two-column data table. Consequently, the model will confidently draw a bounding box around regular paragraphs and delete them.

3. Image Resolution and DPI Degradation

At 72 DPI, small or dense text (such as financial footnotes, superscript numbers, or tight gridlines) becomes heavily pixelated and blurred. The AI struggles to parse these low-fidelity regions, resulting in misclassifications or dropped confidence scores.

Approach #3

Natural Language Processing (NLP)

In [ ]:
!pip install spacy transformers torch
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 57.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import fitz
import re
import spacy
from transformers import pipeline

1. Filtering out tables

In [ ]:
num_pages = min(len(doc), 25)
full_clean_text = ""

for page_num in range(num_pages):
    print(f"  -> Extracting Page {page_num + 1}/{num_pages}...")
    page_text = doc[page_num].get_text("text")

    cleaned_lines = []
# Use heuristic logic to skip lines that look like tables
    for line in page_text.split('\n'):
        numbers = re.findall(r"\d+", line)
        words = re.findall(r"[a-zA-Z]+", line)
        digit_ratio = sum(c.isdigit() for c in line) / max(len(line), 1)

 # If the line triggers any of these rules, it's a table row. Skip it.
        is_table = (
            len(numbers) >= 3 or
            digit_ratio > 0.3 or
            (len(numbers) >= 2 and len(words) >= 2) or
            len(line.split()) > 10 or
            "  " in line
        )

        if not is_table and line.strip():
            cleaned_lines.append(line.strip())

  # Stitch the surviving (non-table) lines together with spaces
    full_clean_text += " ".join(cleaned_lines) + " "

# Final cleanup to remove double spaces
full_clean_text = re.sub(r'\s+', ' ', full_clean_text)


  -> Extracting Page 1/25...
  -> Extracting Page 2/25...
  -> Extracting Page 3/25...
  -> Extracting Page 4/25...
  -> Extracting Page 5/25...
  -> Extracting Page 6/25...
  -> Extracting Page 7/25...
  -> Extracting Page 8/25...
  -> Extracting Page 9/25...
  -> Extracting Page 10/25...
  -> Extracting Page 11/25...
  -> Extracting Page 12/25...
  -> Extracting Page 13/25...
  -> Extracting Page 14/25...
  -> Extracting Page 15/25...
  -> Extracting Page 16/25...
  -> Extracting Page 17/25...
  -> Extracting Page 18/25...
  -> Extracting Page 19/25...
  -> Extracting Page 20/25...
  -> Extracting Page 21/25...
  -> Extracting Page 22/25...
  -> Extracting Page 23/25...
  -> Extracting Page 24/25...
  -> Extracting Page 25/25...


2. LINGUISTIC ANALYSIS (spaCy)


In [ ]:
print("\n2. Running spaCy for Named Entity Recognition...")
nlp = spacy.load("en_core_web_sm")

# Process a chunk of the text to keep memory usage safe
doc_spacy = nlp(full_clean_text[:5000])

print("\n--- Key Financial Entities Found ---")
for ent in doc_spacy.ents:
    if ent.label_ in ["MONEY", "ORG", "DATE"]:
        print(f"{ent.text:25} | {ent.label_}")


# --- 3. TOPIC CLASSIFICATION (Zero-Shot) ---
print("\n3. Loading Classification Model...")
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Grab a random 200-character chunk from the middle of your clean text to test
sample_sentence = full_clean_text[2000:2200]
labels = ["Climate & ESG", "Financial Performance", "Corporate Governance", "Cybersecurity"]

result = classifier(sample_sentence, labels)
print("\n--- Topic Classification ---")
print(f"Text snippet: '{sample_sentence}...'")
print(f"Top Category: {result['labels'][0]} (Confidence: {result['scores'][0]*100:.1f}%)")


2. Running spaCy for Named Entity Recognition...

--- Key Financial Entities Found ---
HSBC                      | ORG
this year                 | DATE
Group                     | ORG
Group                     | ORG
Financial overview Board  | ORG
Environmental Social Governance Financial review | ORG
Financial                 | ORG
Group                     | ORG
Brendan Nelson Group      | ORG
ESG                       | ORG
ESG                       | ORG
Financial Disclosures     | ORG
TCFD                      | ORG
IFRS Accounting Standards | ORG
non-IFRS Accounting Standards | ORG
European Securities and Markets Authority | ORG
US Securities and Exchange Commission | ORG
2025                      | DATE
HSBC Holdings plc         | ORG
2025                      | DATE
2025                      | DATE
Group                     | ORG
2025                      | DATE
2025 full-year            | DATE
RoTE                      | ORG
RoTE                      | ORG
Constant           

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]


--- Topic Classification ---
Text snippet: ' contained on such websites, are incorporated by reference in this report. HSBC Holdings plc Annual Report and Accounts 2025 TCFD Performance in 2025 Our key performance indicators measure the progres...'
Top Category: Financial Performance (Confidence: 69.7%)


4. FINANCIAL SENTIMENT ANALYSIS (FinBERT)

Sentence 1: Pulled from the "Key themes of 2021" section on Page 2.

Sentence 2: Pulled from the "Group Chief Executive’s review" on Page 10.

Sentence 3: Pulled from the "Our global reach" section on Page 6 (slightly paraphrased for length).

In [ ]:
print("\n4. Loading FinBERT Sentiment Model...")
fin_sentiment = pipeline("sentiment-analysis", model="ProsusAI/finbert")

# Analyzing three sentences manually extracted from your text stream
# (In a real pipeline, you would loop this over doc_spacy.sents)
test_sentences = [
    "Performance reflected an improvement in global economic conditions, which resulted in releases of expected credit loss allowances.",
    "Adjusted revenue was down 3%, due mainly to the impact of interest rate cuts.",
    "We maintained a strong capital, funding and liquidity position with a diversified business model."
]

print("\n--- Financial Sentiment Analysis ---")
for sent in test_sentences:
    res = fin_sentiment(sent)[0]
    print(f"[{res['label']:^10}] | {sent}")

print("\nPipeline Complete")


4. Loading FinBERT Sentiment Model...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


--- Financial Sentiment Analysis ---
[ positive ] | Performance reflected an improvement in global economic conditions, which resulted in releases of expected credit loss allowances.
[ negative ] | Adjusted revenue was down 3%, due mainly to the impact of interest rate cuts.
[ positive ] | We maintained a strong capital, funding and liquidity position with a diversified business model.

Pipeline Complete


In [ ]:
print(full_clean_text[:3000])

 Opening up a world of opportunity HSBC is one of the largest banking and financial services organisations in the world. Guided by our purpose of opening up a world of opportunity, our ambition is to become the world’s most trusted bank globally, putting customers at the heart of everything we do. In this year’s report Strategic report Highlights Who we are Group Chairman’s shareholder letter Group CEO’s shareholder letter Our strategy Financial overview Board engagement with stakeholders and decision making Long-term viability and going concern statement Environmental, social and governance (‘ESG’) review Environmental Social Governance Financial review Financial summary Business segments and legal Alternative performance measures Risk review Our approach to risk Corporate governance report Financial statements Additional information This Strategic Report was approved by the behalf by the Group Chairman. Brendan Nelson Group Chairman A reminder The currency we report in is US dollars.

In [ ]:
with open("clean_hsbc_text.txt", "w", encoding="utf-8") as f:
    f.write(full_clean_text)

print("\n--- FILE SAVED ---")


--- FILE SAVED ---
